In [1]:
print("Hello")

Hello


In [ ]:
import time
from huggingface_hub import snapshot_download

for attempt in range(50):
    try:
        print(f"Connecting to Hugging Face... (attempt {attempt+1})")
        snapshot_download(
            repo_id="ai4bharat/MSMARCO-XI",
            repo_type="dataset",
            local_dir="../MSMARCO-XI",
            max_workers=8,
            etag_timeout=30,
        )
        print("All 55GB downloaded successfully!")
        break
    except Exception as e:
        print(f"Connection issue: {e}")
        print("Retrying in 10 seconds (it will resume where it stopped)...")
        time.sleep(10)

In [7]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="ai4bharat/MSMARCO-XI",
    filename="train/teltrain.jsonl",
    repo_type="dataset",
    local_dir="../MSMARCO-XI",
)
print(f"Downloaded to: {path}")

EntryNotFoundError: 404 Client Error. (Request ID: Root=1-6a7da6a1-3774704764e97c36188a6986;57c7a934-cd8b-4066-8b0f-5261f5941c90)

Entry Not Found for url: https://huggingface.co/datasets/ai4bharat/MSMARCO-XI/resolve/main/train/teltrain.jsonl.

In [8]:
from datasets import load_dataset
telugu_train = load_dataset("ai4bharat/IndicMSMARCO", "te")
print(telugu_train)

README.md: 0.00B [00:00, ?B/s]

D:\mlenv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DIBYAJYOTI\.cache\huggingface\hub\datasets--ai4bharat--IndicMSMARCO. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


te/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['query_id', 'query', 'passage', 'passage_id', 'language', 'answer', 'title', 'url', 'query_type', 'relevance_score', 'is_selected', 'text', 'meta', 'dataset', 'source'],
        num_rows: 1000
    })
})


In [21]:
import pyarrow.parquet as pq
import pandas as pd

# Read only the schema first (no memory cost) to check columns
parquet_file = pq.ParquetFile("./MSMARCO-XI/train/pantrain.parquet")
print(parquet_file.schema)

required group field_id=-1 schema {
  optional binary field_id=-1 source_lang (String);
  optional binary field_id=-1 target_lang (String);
  optional group field_id=-1 meta {
    optional int64 field_id=-1 frequency_penalty;
    optional int64 field_id=-1 max_tokens;
    optional binary field_id=-1 model_name (String);
    optional int64 field_id=-1 presence_penalty;
    optional int64 field_id=-1 temperature;
    optional int64 field_id=-1 top_p;
  }
  optional binary field_id=-1 Answer (String);
  optional int64 field_id=-1 query_id;
  optional binary field_id=-1 query_type (String);
  optional group field_id=-1 passages {
    optional group field_id=-1 English_passages (List) {
      repeated group field_id=-1 list {
        optional binary field_id=-1 element (String);
      }
    }
    optional group field_id=-1 Translated_passages (List) {
      repeated group field_id=-1 list {
        optional binary field_id=-1 element (String);
      }
    }
    optional group field_id=-1 is

In [26]:
parquet_file.metadata

  created_by: parquet-cpp-arrow version 19.0.1
  num_columns: 17
  num_rows: 778638
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 10458

In [27]:
parquet_file.schema

required group field_id=-1 schema {
  optional binary field_id=-1 source_lang (String);
  optional binary field_id=-1 target_lang (String);
  optional group field_id=-1 meta {
    optional int64 field_id=-1 frequency_penalty;
    optional int64 field_id=-1 max_tokens;
    optional binary field_id=-1 model_name (String);
    optional int64 field_id=-1 presence_penalty;
    optional int64 field_id=-1 temperature;
    optional int64 field_id=-1 top_p;
  }
  optional binary field_id=-1 Answer (String);
  optional int64 field_id=-1 query_id;
  optional binary field_id=-1 query_type (String);
  optional group field_id=-1 passages {
    optional group field_id=-1 English_passages (List) {
      repeated group field_id=-1 list {
        optional binary field_id=-1 element (String);
      }
    }
    optional group field_id=-1 Translated_passages (List) {
      repeated group field_id=-1 list {
        optional binary field_id=-1 element (String);
      }
    }
    optional group field_id=-1 is

In [22]:
# Read only first N rows using row groups (much lighter on memory)
batch = next(parquet_file.iter_batches(batch_size=2000))
hindi_train_df = batch.to_pandas()
print(hindi_train_df.shape)
print(hindi_train_df.columns.tolist())

(2000, 10)
['source_lang', 'target_lang', 'meta', 'Answer', 'query_id', 'query_type', 'passages', 'Eng_Query', 'Eng_Answer', 'query']


In [37]:
print(hindi_train_df["query_type"].value_counts())

query_type
DESCRIPTION    1325
NUMERIC         387
ENTITY          179
LOCATION         60
PERSON           49
Name: count, dtype: int64


In [38]:
print(hindi_train_df["query_type"].isna().sum())
print(hindi_train_df.shape[0])  # total rows for comparison

0
2000


In [23]:
# Inspect one full row
row = hindi_train_df.iloc[0]
print("Query:", row["query"])
print("Answer:", row["Answer"])
print("Eng_Query:", row["Eng_Query"])
print("Eng_Answer:", row["Eng_Answer"])
print("query_type:", row["query_type"])
print()
print("Passages structure:")
print(type(row["passages"]))
print(row["passages"])

Query: ਮੈਨਹੈਟਨ ਪ੍ਰੋਜੈਕਟ ਦੀ ਸਫਲਤਾ ਦਾ ਤੁਰੰਤ ਪ੍ਰਭਾਵ ਕੀ ਸੀ?
Answer: ਮੈਨਹੈਟਨ ਪ੍ਰੋਜੈਕਟ ਦੀ ਸਫਲਤਾ ਦਾ ਤੁਰੰਤ ਪ੍ਰਭਾਵ ਪਰਮਾਣੂ ਖੋਜਕਰਤਾਵਾਂ ਅਤੇ ਇੰਜੀਨੀਅਰਾਂ ਦੀ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਪ੍ਰਾਪਤੀ ਉੱਤੇ ਇਕਲੌਤਾ ਬੱਦਲ ਸੀ ਜੋ ਉਨ੍ਹਾਂ ਦੀ ਸਫਲਤਾ ਦਾ ਅਸਲ ਅਰਥ ਸੀ; ਲੱਖਾਂ ਬੇਕਸੂਰ ਜੀਵਨ ਨੂੰ ਖਤਮ ਕਰ ਦਿੱਤਾ ਗਿਆ।
Eng_Query: )what was the immediate impact of the success of the manhattan project?
Eng_Answer: The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.
query_type: DESCRIPTION

Passages structure:
<class 'dict'>
{'English_passages': array(['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives

In [36]:
from pathlib import Path
train_dir = Path("../MSMARCO-XI/train")

for parquet_file in train_dir.glob("*.parquet"):
    pf = pq.ParquetFile(parquet_file)
    batch = next(pf.iter_batches(batch_size=2))
    diff_train_df = batch.to_pandas()
    row = diff_train_df.iloc[0]
    print("Name: ",parquet_file.name,"Eng_Query:", row["Eng_Query"])
    print("query_id:", row["query_id"])

Name:  asmtrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869
Name:  bentrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869
Name:  gujtrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869
Name:  hintrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869
Name:  kantrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869
Name:  maltrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869
Name:  martrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869
Name:  neptrain.parquet Eng_Query: )what was the immediate impact of the success of the manhattan project?
query_id: 1185869


In [35]:
from pathlib import Path
train_dir = Path("../MSMARCO-XI/validation")

for parquet_file in train_dir.glob("*.parquet"):
    pf = pq.ParquetFile(parquet_file)
    batch = next(pf.iter_batches(batch_size=2))
    diff_train_df = batch.to_pandas()
    row = diff_train_df.iloc[0]
    print("Name: ",parquet_file.name, "Eng_Query:", row["Eng_Query"])
    print("query_id:", row["query_id"])

Name:  asmval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  benval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  gujval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  hinval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  kanval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  malval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  marval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  nepval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  orival.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  panval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  sanval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  tamval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  telval.parquet Eng_Query: . what is a corporation?
query_id: 1102432
Name:  urdva

In [ ]:
https://excalidraw.com/#json=dDvepZ2OXQjQDcrHtG3Tw,mb_0RCzyrvMYMUnKK4409w

In [46]:
import pyarrow.parquet as pq
from collections import Counter

# parquet_file = pq.ParquetFile("./MSMARCO-XI/train/oritrain.parquet")
#
# query_type_counts = Counter()

def fill_details(parquet_file):
    total_rows = 0
    query_type_counts = Counter()
    for batch in parquet_file.iter_batches(batch_size=20000, columns=["query_type"]):
        df_chunk = batch.to_pandas()
        query_type_counts.update(df_chunk["query_type"].value_counts().to_dict())
        total_rows += len(df_chunk)
        print(f"Processed {total_rows} rows so far...", end="\r")

    print(f"\n\nTotal rows: {total_rows}")
    print(query_type_counts)

In [ ]:
from pathlib import Path
train_dir = Path("../MSMARCO-XI/train")

for file in train_dir.glob("*.parquet"):
    print(f"file name: {file.name}")
    parquet_file = pq.ParquetFile(file)
    fill_details(parquet_file)


| File | Total Rows | DESCRIPTION | NUMERIC | ENTITY | LOCATION | PERSON |
|---|---|---|---|---|---|---|
| asmtrain.parquet | 778,638 | 411,657 (52.9%) | 205,118 (26.3%) | 69,047 (8.9%) | 48,928 (6.3%) | 43,888 (5.6%) |
| bentrain.parquet | 778,638 | 411,657 (52.9%) | 205,118 (26.3%) | 69,047 (8.9%) | 48,928 (6.3%) | 43,888 (5.6%) |
| gujtrain.parquet | 778,638 | 411,657 (52.9%) | 205,118 (26.3%) | 69,047 (8.9%) | 48,928 (6.3%) | 43,888 (5.6%) |
| hintrain.parquet | 778,638 | 411,657 (52.9%) | 205,118 (26.3%) | 69,047 (8.9%) | 48,928 (6.3%) | 43,888 (5.6%) |
| kantrain.parquet | 778,638 | 411,657 (52.9%) | 205,118 (26.3%) | 69,047 (8.9%) | 48,928 (6.3%) | 43,888 (5.6%) |
| maltrain.parquet | 778,638 | 411,657 (52.9%) | 205,118 (26.3%) | 69,047 (8.9%) | 48,928 (6.3%) | 43,888 (5.6%) |
| martrain.parquet | 765,873 | 404,697 (52.8%) | 200,853 (26.2%) | 68,167 (8.9%) | 48,529 (6.3%) | 43,627 (5.7%) |
| neptrain.parquet | 754,154 | 398,470 (52.8%) | 196,963 (26.1%) | 67,207 (8.9%) | 48,144 (6.4%) | 43,370 (5.8%) |
| oritrain.parquet | 782,282 | 413,418 (52.8%) | 205,972 (26.3%) | 69,405 (8.9%) | 49,306 (6.3%) | 44,181 (5.6%) |
| pantrain.parquet | 778,638 | 411,657 (52.9%) | 205,118 (26.3%) | 69,047 (8.9%) | 48,928 (6.3%) | 43,888 (5.6%) |


In [1]:
import re
import uuid
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

In [2]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

In [3]:
client

In [4]:
def stream_read_parquet(path: str, batch_size: int = 5000) -> pd.DataFrame:
    """TO avoid memory error we use this batch wise system"""
    pf = pq.ParquetFile(path)
    batches = []
    for batch in pf.iter_batches(batch_size=batch_size):
        batches.append(batch.to_pandas())
    return pd.concat(batches, ignore_index=True)

In [5]:
# Stratified subsample by query_type: It is for manging query type dist in between DESCRIPTION, NUMERIC, ENTITY,LOCATION , PERSON and the percentage is taken from upper table.

QUERY_TYPE_DIST = {
    "DESCRIPTION": 0.529,
    "NUMERIC": 0.263,
    "ENTITY": 0.089,
    "LOCATION": 0.063,
    "PERSON": 0.056,
}

def stratified_sample(df: pd.DataFrame, n_total: int, seed: int = 42) -> pd.DataFrame:
    parts = []
    for qtype, frac in QUERY_TYPE_DIST.items():
        n = int(round(n_total * frac))
        pool = df[df["query_type"] == qtype]
        if len(pool) == 0:
            continue
        n = min(n, len(pool))
        parts.append(pool.sample(n=n, random_state=seed))
    return pd.concat(parts, ignore_index=True) if parts else df.sample(
        n=min(n_total, len(df)), random_state=seed
    )

In [6]:
def explode_msmarco_xi(df: pd.DataFrame, lang_code: str) -> pd.DataFrame:
    """MSMARCO-XI: nested `passages` dict with parallel arrays."""
    rows = []
    for _, r in df.iterrows():
        p = r["passages"]
        eng_passages = p["English_passages"]
        trans_passages = p["Translated_passages"]
        is_sel = p["is_selected"]
        for i, (en_p, tr_p, sel) in enumerate(zip(eng_passages, trans_passages, is_sel)):
            rows.append({
                "text": tr_p,
                "text_en": en_p,
                "lang": lang_code,
                "query": r["query"],
                "query_en": r["Eng_Query"],
                "answer": r["Answer"],
                "answer_en": r["Eng_Answer"],
                "query_id": int(r["query_id"]),
                "passage_id": f"{lang_code}_{r['query_id']}_{i}",
                "source_dataset": "MSMARCO-XI",
                "is_selected": bool(sel),
                "query_type": r["query_type"],
            })
    return pd.DataFrame(rows)

In [7]:
YEAR_RE = re.compile(r"\b(1[89]\d{2}|20\d{2})\b")

def enrich(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich the dataset by using regex years and spaCy NER (on text_en only)"""
    import spacy
    nlp = spacy.load("en_core_web_sm")

    year_mentions, ppl, orgs, locs = [], [], [], []
    for doc_text in nlp.pipe(df["text_en"].fillna("").tolist(), batch_size=256):
        year_mentions.append([int(y) for y in YEAR_RE.findall(doc_text.text)])
        ppl.append([e.text for e in doc_text.ents if e.label_ == "PERSON"])
        orgs.append([e.text for e in doc_text.ents if e.label_ == "ORG"])
        locs.append([e.text for e in doc_text.ents if e.label_ in ("GPE", "LOC")])

    df["year_mentions"] = year_mentions
    df["entities_people"] = ppl
    df["entities_orgs"] = orgs
    df["entities_locations"] = locs
    return df

In [8]:
# Native Chucking style -> (one row one chunk)
# No processing.

def chunk_passage_native(df: pd.DataFrame) -> pd.DataFrame:
    """MSMARCO passages are already ~1 unit each -- use as-is."""
    out = df.copy()
    out["chunk_strategy"] = "passage_native"
    out["chunk_id"] = out["passage_id"] + "_native"
    return out

In [9]:
# Fixed overlap -> it 0.2 overlap starrgy in 256 words (256 <- 0.2 -> 256 <- 0.2 -> 256
# It cares about size, not meaning.

def chunk_fixed_overlap(df: pd.DataFrame, chunk_tokens: int = 256, overlap: float = 0.2) -> pd.DataFrame:
    """Sliding window over text_en, word-approx tokens, 20% overlap."""
    rows = []
    step = int(chunk_tokens * (1 - overlap))
    for _, r in df.iterrows():
        words = (r["text_en"] or "").split()
        if len(words) <= chunk_tokens:
            spans = [words]
        else:
            spans = [words[i:i + chunk_tokens] for i in range(0, len(words), step) if words[i:i + chunk_tokens]]
        for j, span in enumerate(spans):
            new_row = r.to_dict()
            new_row["text_en"] = " ".join(span)
            new_row["chunk_strategy"] = "fixed_overlap"
            new_row["chunk_id"] = f"{r['passage_id']}_fixed_{j}"
            rows.append(new_row)
    return pd.DataFrame(rows)

In [10]:
# Semantic Chucking - It cares about meaning/topic boundaries.

def chunk_semantic(df: pd.DataFrame, model, similarity_threshold: float = 0.75) -> pd.DataFrame:
    """Split at sentence-embedding similarity breakpoints."""
    import re as _re
    rows = []
    for _, r in df.iterrows():
        sents = [s.strip() for s in _re.split(r"(?<=[.!?])\s+", r["text_en"] or "") if s.strip()]
        if len(sents) <= 1:
            groups = [sents]
        else:
            embs = model.encode(sents, normalize_embeddings=True)
            groups, current = [], [sents[0]]
            for i in range(1, len(sents)):
                sim = float(np.dot(embs[i - 1], embs[i]))
                if sim < similarity_threshold:
                    groups.append(current)
                    current = []
                current.append(sents[i])
            groups.append(current)
        for j, g in enumerate(groups):
            new_row = r.to_dict()
            new_row["text_en"] = " ".join(g)
            new_row["chunk_strategy"] = "semantic"
            new_row["chunk_id"] = f"{r['passage_id']}_sem_{j}"
            rows.append(new_row)
    return pd.DataFrame(rows)


```text
                    RAW DATA
                       │
                       ▼
              explode / normalize
                       │
                       ▼
                    enrich()
                       │
          ┌────────────┼─────────────┐
          ▼            ▼             ▼
       years        people        organizations
                       │
                       ▼
             chunking strategies
          ┌────────────┼─────────────┐
          ▼            ▼             ▼
       native        fixed        semantic
          └────────────┼─────────────┘
                       ▼
              tag_metadata_aware()
                       │
                       ▼
              has_year / has_person
              has_org / has_location
                       │
                       ▼
                unified dataset
```


In [11]:
def tag_metadata_aware(df: pd.DataFrame) -> pd.DataFrame:
    """Not a separate split -- tags chunks (any strategy) for payload filtering.
    Applied to all chunk variants before writing the unified file."""
    df = df.copy()
    df["has_year"] = df["year_mentions"].apply(lambda y: len(y) > 0)
    df["has_person"] = df["entities_people"].apply(lambda p: len(p) > 0)
    df["has_org"] = df["entities_orgs"].apply(lambda o: len(o) > 0)
    df["has_location"] = df["entities_locations"].apply(lambda l: len(l) > 0)
    return df

In [12]:
OUTPUT_FILE = "../MSMARCO-XI/unified_corpus.parquet"
QDRANT_COLLECTION = "msmarco_english_corpus"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
EMBED_DIM = 384

def write_unified_file(df: pd.DataFrame, path: str = OUTPUT_FILE):
    df.to_parquet(path, index=False)
    print(f"Wrote {len(df):,} rows to {path}")

In [13]:
# Take your final text chunks then convert them into embeddings then store those embeddings and metadata in Qdrant.

def embed_and_upsert(df: pd.DataFrame, qdrant_url: str = "http://localhost:6333"):
    from sentence_transformers import SentenceTransformer
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct

    model = SentenceTransformer(EMBED_MODEL)
    client = QdrantClient(url=qdrant_url)

    client.recreate_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )

    texts = df["text_en"].fillna("").tolist()
    vectors = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

    points = []
    for vec, (_, row) in zip(vectors, df.iterrows()):
        payload = row.drop(labels=["text_en"]).to_dict()
        payload["text_en"] = row["text_en"]
        points.append(
            PointStruct(id=str(uuid.uuid4()), vector=vec.tolist(), payload=payload)
        )

    # batch upsert
    B = 256
    for i in range(0, len(points), B):
        client.upsert(collection_name=QDRANT_COLLECTION, points=points[i:i + B])

    print(f"Upserted {len(points):,} points into '{QDRANT_COLLECTION}'")


In [14]:
hi_path = "../MSMARCO-XI/train/hintrain.parquet"

hi_df = stream_read_parquet(hi_path)

In [15]:
print(f"Original Hindi rows: {len(hi_df):,}")

Original Hindi rows: 778,638


In [16]:
SAMPLE_SIZE_PER_LANG = 12_000

hi_df = stratified_sample(hi_df, SAMPLE_SIZE_PER_LANG)

In [17]:
hi_norm = explode_msmarco_xi(hi_df, "hi")
print(f"After explosion: {len(hi_norm):,}")

After explosion: 2,007


In [18]:
unified = pd.concat([hi_norm], ignore_index=True)
print(f"After unification: {len(unified):,}")
print(f"Unified done")


After unification: 2,007
Unified done


In [19]:
unified = enrich(unified)

In [20]:
native = chunk_passage_native(unified)
fixed = chunk_fixed_overlap(unified)
print(f"After fixed overlap: {len(fixed):,}")
print(f"After fixed overlap: {len(fixed):,}")

After fixed overlap: 2,007
After fixed overlap: 2,007


In [21]:
from sentence_transformers import SentenceTransformer
sem_model = SentenceTransformer(EMBED_MODEL)
semantic = chunk_semantic(unified, sem_model)
print(f"After semantic: {len(semantic):,}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\mlenv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DIBYAJYOTI\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

After semantic: 5,754


In [22]:
all_chunks = pd.concat([native, fixed, semantic], ignore_index=True)
all_chunks = tag_metadata_aware(all_chunks)
print(f"After tag aware: {len(all_chunks):,}")

After tag aware: 9,768


In [23]:
write_unified_file(all_chunks)

Wrote 9,768 rows to unified_corpus.parquet


In [24]:
embed_and_upsert(all_chunks)

D:\Users\Dibyajyoti\Temp\Temp\ipykernel_19320\1858209814.py:11: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Upserted 9,768 points into 'msmarco_english_corpus'


In [26]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
QDRANT_COLLECTION = "msmarco_english_corpus"

model = SentenceTransformer(EMBED_MODEL)

client = QdrantClient(
    url="http://localhost:6333"
)

query = "Who invented the mechanical calculator?"

query_vector = model.encode(
    query,
    normalize_embeddings=True
).tolist()

results = client.query_points(
    collection_name=QDRANT_COLLECTION,
    query=query_vector,
    limit=5,
    with_payload=True
)

for i, result in enumerate(results.points, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", result.score)
    print("Text:", result.payload["text_en"])
    print("Passage ID:", result.payload["passage_id"])
    print("Chunk:", result.payload["chunk_strategy"])


--- Result 1 ---
Score: 0.8645128
Text: In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.
Passage ID: hi_1003697_0
Chunk: semantic

--- Result 2 ---
Score: 0.8645128
Text: In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.
Passage ID: hi_1003697_0
Chunk: passage_native

--- Result 3 ---
Score: 0.8645128
Text: In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.
Passage ID: 

In [27]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
import time

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
QDRANT_COLLECTION = "msmarco_english_corpus"

# Load ONCE
model = SentenceTransformer(EMBED_MODEL)

# Connect ONCE
client = QdrantClient(
    url="http://localhost:6333"
)


def retrieve(
    query: str,
    top_k: int = 5,
):
    start = time.perf_counter()

    # 1. Encode query
    query_vector = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    encode_time = time.perf_counter()

    # 2. Search Qdrant
    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector.tolist(),
        limit=top_k,
        with_payload=True,
    )

    search_time = time.perf_counter()

    print(f"Embedding: {(encode_time - start) * 1000:.2f} ms")
    print(f"Qdrant:    {(search_time - encode_time) * 1000:.2f} ms")
    print(f"Total:     {(search_time - start) * 1000:.2f} ms")

    return results.points


results = retrieve(
    "Who invented the mechanical calculator?",
    top_k=5
)

for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result.score:.4f}")
    print(result.payload["text_en"])

Embedding: 149.51 ms
Qdrant:    25.75 ms
Total:     175.26 ms

1. Score: 0.8645
In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.

2. Score: 0.8645
In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.

3. Score: 0.8645
In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.

4. Score: 0.7915
Blaise Pascal invented the first digital calculator in 1642 at the age of 16 in France to

In [28]:
def retrieve(query: str, top_k: int = 5, fetch_k: int = 20):

    query_vector = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector.tolist(),
        limit=fetch_k,
        with_payload=True,
    )

    unique = []
    seen = set()

    for result in results.points:
        text = result.payload.get("text_en", "").strip()

        if not text:
            continue

        if text in seen:
            continue

        seen.add(text)
        unique.append(result)

        if len(unique) >= top_k:
            break

    return unique

In [29]:
results = retrieve(
    "Who invented the telephone?",
    top_k=5
)

for i, result in enumerate(results, 1):
    print(
        f"\n{i}. "
        f"Score: {result.score:.4f}"
    )
    print(result.payload["text_en"])


1. Score: 0.6759
In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.

2. Score: 0.6545
Blaise Pascal invented the first digital calculator in 1642 at the age of 16 in France to help his father with tax collection. His device was also called the Pascal's calculator or the Pascaline or the Arithmetique.

3. Score: 0.6266
The calculators that we use today are the advanced versions of a device that, when invented, was in its primitive form and was called the adding machine. Abacus. THE abacus wasn’t a machine but a tool used for counting that was invented before 2000BC. Considered the basis on which the modern calculator has taken its shape, an abacus consisted of small ball-shaped beads that slid on sticks and the sticks were attached to a rectangular frame.

4. Score: 0.6225
2  The f

In [30]:
results = retrieve(
    "Who invented the mechanical calculator?",
    top_k=5
)

for i, result in enumerate(results, 1):
    print(
        f"\n{i}. "
        f"Score: {result.score:.4f}"
    )
    print(result.payload["text_en"])


1. Score: 0.8645
In 1642, during the Renaissance era, Blaise Pascal invented the mechanical calculator which was the first device to perform addition, subtraction, multiplication and division on its own – an advanced version of abacus, but a primitive form of the calculator.

2. Score: 0.7915
Blaise Pascal invented the first digital calculator in 1642 at the age of 16 in France to help his father with tax collection. His device was also called the Pascal's calculator or the Pascaline or the Arithmetique.

3. Score: 0.7884
2  The first digital calculators were invented by Blaise Pascal and he worked for three years on the calculator’s that is from 1643 to 1645.

4. Score: 0.7839
The first mechanical calculator appeared in 1642, the creations of French intellectual and mathematics whizz kid Blaise Pascal as a device that will eventually perform all four arithmetic operations without relying on human intelligence.

5. Score: 0.7480
1 Calculator is nothing but a tiny electronic device use

In [31]:
import time

def retrieve(query: str, top_k: int = 5, fetch_k: int = 20):

    total_start = time.perf_counter()

    # --------------------------------------------------
    # 1. Encode query
    # --------------------------------------------------
    embed_start = time.perf_counter()

    query_vector = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    embed_end = time.perf_counter()

    # --------------------------------------------------
    # 2. Qdrant search
    # --------------------------------------------------
    qdrant_start = time.perf_counter()

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector.tolist(),
        limit=fetch_k,
        with_payload=True,
    )

    qdrant_end = time.perf_counter()

    # --------------------------------------------------
    # 3. Deduplicate
    # --------------------------------------------------
    dedup_start = time.perf_counter()

    unique = []
    seen = set()

    for result in results.points:
        text = result.payload.get("text_en", "").strip()

        if not text:
            continue

        if text in seen:
            continue

        seen.add(text)
        unique.append(result)

        if len(unique) >= top_k:
            break

    dedup_end = time.perf_counter()

    total_end = time.perf_counter()

    # --------------------------------------------------
    # Timing logs
    # --------------------------------------------------
    print("\n========== RETRIEVAL TIMING ==========")
    print(f"Query embedding : {(embed_end - embed_start) * 1000:.2f} ms")
    print(f"Qdrant search   : {(qdrant_end - qdrant_start) * 1000:.2f} ms")
    print(f"Deduplication   : {(dedup_end - dedup_start) * 1000:.2f} ms")
    print(f"Total retrieval : {(total_end - total_start) * 1000:.2f} ms")
    print("======================================")

    return unique

In [64]:
results = retrieve(
    "when is raising the minimum wage bad",
    top_k=5
)

for i, result in enumerate(results, 1):
    print(
        f"\n{i}. "
        f"Score: {result.score:.4f}"
    )
    print(result.payload["text_en"])


========== RETRIEVAL TIMING ==========
Query embedding : 59.74 ms
Qdrant search   : 21.35 ms
Deduplication   : 0.02 ms
Total retrieval : 81.10 ms

1. Score: 0.8461
by Veronique de Rugy. We heard many bad and tired ideas during last night’s State of the Union address, and one of them was the promise to raise the minimum wage, to $9 dollar an hour.eumark, who has done extensive research on the issue, summarizes his results the following way: “Based on 20 years of research, I doubt there is ever a good time to raise the minimum wage.” And the negative consequences are worse when unemployment is high.

2. Score: 0.8434
We heard many bad and tired ideas during last night’s State of the Union address, and one of them was the promise to raise the minimum wage, to $9 dollar an hour.eumark, who has done extensive research on the issue, summarizes his results the following way: “Based on 20 years of research, I doubt there is ever a good time to raise the minimum wage.” And the negative consequ

In [65]:
# First 200 Hindi source rows
hi_df_200 = stratified_sample(hi_df, 200)

# Get the English questions from those 200 rows
test_queries = (
    hi_df_200["Eng_Answer"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(f"Found {len(test_queries)} unique English queries\n")

for i, q in enumerate(test_queries, 1):
    print(f"{i}. {q}")

Found 118 unique English queries

1. Yes,record labels controls the release of albums.
2. Because they have a legal identity separate from those of their owners.
3. No Answer Present.
4. Abraham Lincoln and it's given as a young man.
5. Eye twitching, also known as eye spasm, eyelid spasm, or eye muscle twitch is characterized by the involuntary contraction of the muscles around the eyelids, which further results in blinking of the eyes.
6. A pink slip must be given to an employee who is terminated while under contract and is part of a collective bargain agreement or a union.
7. It is a substance consisting of atoms which all have the same number of protons-i.e. the same atomic number.
8. Brth defects
9. TACACS+
10. Calories
11. +3
12. To take great pleasure or delight.
13. Yes
14. Potentially reduce the risk of developing heart disease, promote healthy weight loss, and improve bone density.
15. It is the informal name for the European Global Navigation Satellite System.
16. Waste mana

In [51]:
from qdrant_client.models import (
    Filter,
    FieldCondition,
    MatchValue,
)


def retrieve_metadata(
    query: str,
    top_k: int = 5,
    has_year: bool | None = None,
    has_person: bool | None = None,
    has_org: bool | None = None,
    has_location: bool | None = None,
    lang: str | None = None,
    chunk_strategy: str | None = None,
):
    """
    Semantic retrieval + optional metadata filtering.
    """

    # --------------------------------------------------
    # 1. Encode query
    # --------------------------------------------------
    query_vector = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    # --------------------------------------------------
    # 2. Build metadata filters
    # --------------------------------------------------
    conditions = []

    if has_year is not None:
        conditions.append(
            FieldCondition(
                key="has_year",
                match=MatchValue(value=has_year),
            )
        )

    if has_person is not None:
        conditions.append(
            FieldCondition(
                key="has_person",
                match=MatchValue(value=has_person),
            )
        )

    if has_org is not None:
        conditions.append(
            FieldCondition(
                key="has_org",
                match=MatchValue(value=has_org),
            )
        )

    if has_location is not None:
        conditions.append(
            FieldCondition(
                key="has_location",
                match=MatchValue(value=has_location),
            )
        )

    if lang is not None:
        conditions.append(
            FieldCondition(
                key="lang",
                match=MatchValue(value=lang),
            )
        )

    if chunk_strategy is not None:
        conditions.append(
            FieldCondition(
                key="chunk_strategy",
                match=MatchValue(value=chunk_strategy),
            )
        )

    # No filter if nothing specified
    query_filter = Filter(must=conditions) if conditions else None

    # --------------------------------------------------
    # 3. Search Qdrant
    # --------------------------------------------------
    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector.tolist(),
        query_filter=query_filter,
        limit=top_k,
        with_payload=True,
    )

    return results.points

In [56]:
results = retrieve_metadata(
    "where was the first calculator invented",
    top_k=5,
    has_location=True,
)

In [ ]:
results


In [58]:
import time

from qdrant_client.models import (
    Filter,
    FieldCondition,
    MatchValue,
)


def hybrid_retrieve(
    query: str,
    top_k: int = 5,

    # Metadata filters
    has_year: bool | None = None,
    has_person: bool | None = None,
    has_org: bool | None = None,
    has_location: bool | None = None,

    lang: str | None = None,
    chunk_strategy: str | None = None,

    # Fetch more candidates before deduplication
    fetch_k: int = 30,
):
    total_start = time.perf_counter()

    # ==========================================================
    # 1. Query embedding
    # ==========================================================
    embed_start = time.perf_counter()

    query_vector = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    embed_time = time.perf_counter() - embed_start

    # ==========================================================
    # 2. Build metadata filter
    # ==========================================================
    filter_start = time.perf_counter()

    conditions = []

    metadata = {
        "has_year": has_year,
        "has_person": has_person,
        "has_org": has_org,
        "has_location": has_location,
        "lang": lang,
        "chunk_strategy": chunk_strategy,
    }

    for key, value in metadata.items():

        if value is not None:

            conditions.append(
                FieldCondition(
                    key=key,
                    match=MatchValue(value=value),
                )
            )

    query_filter = (
        Filter(must=conditions)
        if conditions
        else None
    )

    filter_time = time.perf_counter() - filter_start

    # ==========================================================
    # 3. Vector + metadata search
    # ==========================================================
    qdrant_start = time.perf_counter()

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,

        # VECTOR SEARCH
        query=query_vector.tolist(),

        # METADATA FILTER
        query_filter=query_filter,

        limit=fetch_k,
        with_payload=True,
    )

    qdrant_time = time.perf_counter() - qdrant_start

    # ==========================================================
    # 4. Deduplicate by passage_id
    # ==========================================================
    dedup_start = time.perf_counter()

    unique_results = []
    seen_passages = set()

    for result in results.points:

        passage_id = result.payload.get("passage_id")

        if passage_id in seen_passages:
            continue

        seen_passages.add(passage_id)

        unique_results.append(result)

        if len(unique_results) >= top_k:
            break

    dedup_time = time.perf_counter() - dedup_start

    total_time = time.perf_counter() - total_start

    # ==========================================================
    # 5. Timing
    # ==========================================================
    print("\n========== HYBRID RETRIEVAL ==========")

    print(f"Embedding      : {embed_time * 1000:.2f} ms")
    print(f"Filter build   : {filter_time * 1000:.2f} ms")
    print(f"Qdrant search  : {qdrant_time * 1000:.2f} ms")
    print(f"Deduplication  : {dedup_time * 1000:.2f} ms")
    print(f"Total          : {total_time * 1000:.2f} ms")

    print(f"Candidates     : {len(results.points)}")
    print(f"Final results  : {len(unique_results)}")

    print("\nMetadata filters:")

    for key, value in metadata.items():

        if value is not None:
            print(f"  {key} = {value}")

    print("======================================")

    return unique_results

In [59]:
results = hybrid_retrieve(
    "where was the first calculator invented?",
    top_k=5,
    has_location=True,
)


========== HYBRID RETRIEVAL ==========
Embedding      : 66.18 ms
Filter build   : 0.15 ms
Qdrant search  : 92.71 ms
Deduplication  : 0.03 ms
Total          : 159.07 ms
Candidates     : 30
Final results  : 5

Metadata filters:
  has_location = True


In [60]:
results = hybrid_retrieve(
    "where was the first calculator invented?",
    top_k=5,
)


========== HYBRID RETRIEVAL ==========
Embedding      : 58.42 ms
Filter build   : 0.01 ms
Qdrant search  : 37.82 ms
Deduplication  : 0.01 ms
Total          : 96.26 ms
Candidates     : 30
Final results  : 5

Metadata filters:


In [61]:
results = hybrid_retrieve(
    "where was the first calculator invented?",
    top_k=5,
    has_location=True,
)


========== HYBRID RETRIEVAL ==========
Embedding      : 75.22 ms
Filter build   : 0.08 ms
Qdrant search  : 65.21 ms
Deduplication  : 0.02 ms
Total          : 140.53 ms
Candidates     : 30
Final results  : 5

Metadata filters:
  has_location = True


In [62]:
results = hybrid_retrieve(
    "Who was the Swedish naturalist who established the modern system for classifying organisms?",
    top_k=5,
    has_person=True,
)


========== HYBRID RETRIEVAL ==========
Embedding      : 112.58 ms
Filter build   : 0.09 ms
Qdrant search  : 83.47 ms
Deduplication  : 0.02 ms
Total          : 196.16 ms
Candidates     : 30
Final results  : 5

Metadata filters:
  has_person = True


In [63]:
for i, result in enumerate(results, 1):
    p = result.payload

    print(
        f"{i}. "
        f"score={result.score:.4f}, "
        f"selected={p.get('is_selected')}, "
        f"person={p.get('entities_people')}"
    )
    print(p.get("text_en"))

1. score=0.8091, selected=False, person=['Carl Von Linne']
in the 1700s the Swedish scientist Carl Von Linne developed a new classification system for living things.
2. score=0.7236, selected=False, person=['Carolus Linnaeus']
Carolus Linnaeus (1707-1778) developed binomial nomenclature, the formal naming of species, as part of his work in the taxonomic classification of living things.
3. score=0.6910, selected=False, person=['Carl Woese', 'Charles Linneaus']
Carl Woese proposed the most recent changes to the … classification system in 1990, introducing three domains, archaea, bacteria, and eucarya, by the type of RNA in their cells.Charles Linneaus created the actual groups of the basis of the modern classification system.-Malia1699.
4. score=0.6244, selected=False, person=['Linnaeus', 'Source(s']
However, binomial nomenclature in various forms existed before Linnaeus, and was used by the Bauhins, who lived nearly two hundred years before Linnaeus.
5. score=0.6176, selected=False, per